In [89]:
import re
import mgrs

import pandas as pd
import geopandas as gpd

from zipfile import ZipFile
from datetime import datetime
from shapely.geometry import Point
from ipyleaflet import Map, GeoData, basemaps, LayersControl

In [90]:
def read_sm_zip(path: str) -> list:
    """
    Takes a path to a zip file containing stridsmeldinger.
    Returns a list of raw stridsmelding strings.
    """
    stridsmeldinger = []
    with ZipFile(path) as sm_zip:
        for file_name in sm_zip.namelist():
            with sm_zip.open(file_name) as sm:
                stridsmeldinger.append(sm.read().decode('utf-8'))
    return stridsmeldinger

def find_substring_between(start: str, stop: str, stridsmelding):
    start = re.escape(start)
    end   = re.escape(stop)
    result = re.search('%s(.*)%s' % (start, end), stridsmelding).group(1)
    return result.strip()

def parse_sm_date(date_str: str) -> datetime:
    return datetime.strptime(date_str, '%d%H%MZ%b%y')

def from_mgrs(coordinate: str) -> Point:
    m = mgrs.MGRS()
    lat, lon = m.toLatLon(coordinate)
    return Point(lon,lat)

def parse_stridsmelding(stridsmelding):
    date_str = find_substring_between("DTG", "\n", stridsmelding)
    sm_datetime = parse_sm_date(date_str)
    
    sm_from = find_substring_between("FRA:", "\n", stridsmelding)
    sm_to = find_substring_between("TIL:", "\n", stridsmelding)
    sm_message = find_substring_between("\n\n", "\n", stridsmelding)

    pos_mgrs = find_substring_between("posisjon", "DTG", sm_message)
    pos_point = from_mgrs(pos_mgrs)
    
    sign = find_substring_between("---", "---", stridsmelding)
    
    return {
        'datetime': sm_datetime,
        'from': sm_from,
        'to': sm_to,
        'message': sm_message,
        'sign': sign,
        'geometry': pos_point
    }

In [91]:
# Les ut alle stridsmeldinger fra zip inn i en liste med strenger.
stridsmeldinger = read_sm_zip('data/stridsmelding.zip')
data = []
for sm in stridsmeldinger:
    data.append(parse_stridsmelding(sm))

In [92]:
df = pd.DataFrame(data)
gdf = gpd.GeoDataFrame(
    df[['from', 'to', 'message', 'sign']],
    geometry=df['geometry']
)

In [93]:
m = Map(
    basemap=basemaps.OpenStreetMap.Mapnik,
    center=(39.96123, 116.36739 ),
    zoom=10
)

geo_data = GeoData(geo_dataframe = gdf,
    name = 'stridsmelding')

m.add(geo_data)
m.add(LayersControl())

m

Map(center=[39.96123, 116.36739], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', …

In [88]:
mgrs.MGRS()

In [95]:
gdf.to_file('stridsmelding.shp')